# THỰC HÀNH MÁY HỌC VỚI PYSPARK

## Mục tiêu học tập


1. Tạo `SparkSession` và đọc dữ liệu bằng DataFrame.
2. Chuẩn bị dữ liệu cho Machine Learning trong Spark.
3. Xử lý missing values, biến đổi cột và categorical features.
4. Dùng `StringIndexer`, `OneHotEncoder`, `VectorAssembler`.
5. Xây dựng Decision Tree và Logistic Regression cho classification.
6. Đánh giá classifier bằng confusion matrix, accuracy, precision, recall, F1 và AUC.
7. Biến văn bản thành vector bằng Tokenizer → StopWordsRemover → HashingTF → IDF.
8. Xây dựng Linear Regression và đánh giá RMSE/MAE/R².
9. Thực hiện feature engineering và bucketing.
10. Hiểu Ridge, Lasso và Elastic Net.
11. Xây dựng `Pipeline` để tránh data leakage.
12. Dùng Cross Validation + Grid Search để chọn hyperparameter.
13. Làm quen với ensemble: Decision Tree, Random Forest, Gradient-Boosted Trees.

<a id="setup"></a>
## 1. Chuẩn bị môi trường

In [36]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("pyspark") is None:
    print("Đang cài PySpark...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark"])
else:
    print("PySpark đã sẵn sàng.")

PySpark đã sẵn sàng.


In [2]:
from pathlib import Path

candidates = [
    Path.cwd() / "data",
    Path("/content/data"),
    Path.cwd().parent / "data"
]
DATA_DIR = next((p for p in candidates if p.exists()), None)

print("DATA_DIR =", DATA_DIR)
if DATA_DIR:
    for p in sorted(DATA_DIR.iterdir()):
        print(" -", p.name)

DATA_DIR = C:\Users\chinhnt0500\Downloads\33241026114_NguyenThiChinh_MLPySpark\data
 - flights-larger.csv
 - sms.csv


<a id="mlspark"></a>
## 1. Machine Learning và Spark

### 1.1. Machine Learning

Công thức bánh waffle:

- cách truyền thống: con người viết quy tắc;
- Machine Learning: máy tính học quy tắc từ nhiều ví dụ.

Hai bài toán cơ bản:

- **Regression**: dự đoán giá trị liên tục.
- **Classification**: dự đoán nhãn/lớp.

### 1.2. Vì sao dùng Spark?

Khi dữ liệu vượt quá khả năng xử lý của một máy, Spark có thể:

- phân tán dữ liệu trên cluster;
- xử lý song song;
- tận dụng bộ nhớ;
- cung cấp API cấp cao cho Python, Scala, Java, R.

Trong phần Machine Learning, ta dùng chủ yếu **`pyspark.ml`**.

<a id="session"></a>
## 2. Tạo SparkSession

```python
SparkSession.builder.master("local[*]").appName(...).getOrCreate()
```

`local[*]` nghĩa là sử dụng các CPU core khả dụng trên máy cục bộ.

In [3]:
import sys
import pyspark

print("Python:", sys.executable)
print("PySpark:", pyspark.__version__)

from pyspark.sql import SparkSession
from pyspark.ml.classification import LogisticRegression

print("Import thành công")

Python: C:\Users\chinhnt0500\AppData\Local\anaconda3\envs\pyspark_bt01\python.exe
PySpark: 4.2.0
Import thành công


In [7]:
import pyspark
from pyspark.sql import SparkSession, functions as F

print("PySpark:", pyspark.__version__, flush=True)

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("MachineLearningWithPySpark")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)

print("Spark:", spark.version)
spark.range(5).show()

PySpark: 4.2.0
Spark: 4.2.0
+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



<a id="loading"></a>
## 3. Đọc và khảo sát dữ liệu

- `flights-larger.csv`: bài toán dự đoán chuyến bay trễ và thời gian bay.
- `sms.csv`: bài toán phân loại SMS spam.

### 3.1. Đọc dữ liệu chuyến bay

In [8]:
flights = spark.read.csv(
    str(DATA_DIR / "flights-larger.csv"),
    header=True,
    inferSchema=True,
    nullValue="NA"
)

print("Số bản ghi:", flights.count())
flights.show(5)
flights.printSchema()

Số bản ghi: 275000
+---+---+---+-------+------+---+----+------+--------+-----+
|mon|dom|dow|carrier|flight|org|mile|depart|duration|delay|
+---+---+---+-------+------+---+----+------+--------+-----+
| 10| 10|  1|     OO|  5836|ORD| 157|  8.18|      51|   27|
|  1|  4|  1|     OO|  5866|ORD| 466|  15.5|     102| NULL|
| 11| 22|  1|     OO|  6016|ORD| 738|  7.17|     127|  -19|
|  2| 14|  5|     B6|   199|JFK|2248| 21.17|     365|   60|
|  5| 25|  3|     WN|  1675|SJC| 386| 12.92|      85|   22|
+---+---+---+-------+------+---+----+------+--------+-----+
only showing top 5 rows
root
 |-- mon: integer (nullable = true)
 |-- dom: integer (nullable = true)
 |-- dow: integer (nullable = true)
 |-- carrier: string (nullable = true)
 |-- flight: integer (nullable = true)
 |-- org: string (nullable = true)
 |-- mile: integer (nullable = true)
 |-- depart: double (nullable = true)
 |-- duration: integer (nullable = true)
 |-- delay: integer (nullable = true)



Các cột chính

- `mon`: tháng
- `dom`: ngày trong tháng
- `dow`: ngày trong tuần
- `carrier`: hãng bay
- `flight`: số hiệu chuyến bay
- `org`: sân bay xuất phát
- `mile`: quãng đường
- `depart`: giờ khởi hành
- `duration`: thời gian bay
- `delay`: độ trễ

### 3.2. Các thao tác kiểm tra DataFrame

In [9]:
print("Columns:", flights.columns)
print("Dtypes :", flights.dtypes)
print("Rows   :", flights.count())

flights.select("carrier", "org", "mile", "duration", "delay").show(10)

Columns: ['mon', 'dom', 'dow', 'carrier', 'flight', 'org', 'mile', 'depart', 'duration', 'delay']
Dtypes : [('mon', 'int'), ('dom', 'int'), ('dow', 'int'), ('carrier', 'string'), ('flight', 'int'), ('org', 'string'), ('mile', 'int'), ('depart', 'double'), ('duration', 'int'), ('delay', 'int')]
Rows   : 275000
+-------+---+----+--------+-----+
|carrier|org|mile|duration|delay|
+-------+---+----+--------+-----+
|     OO|ORD| 157|      51|   27|
|     OO|ORD| 466|     102| NULL|
|     OO|ORD| 738|     127|  -19|
|     B6|JFK|2248|     365|   60|
|     WN|SJC| 386|      85|   22|
|     B6|LGA|1076|     182|   70|
|     B6|ORD| 740|     130|   47|
|     UA|SFO| 679|     123|  135|
|     US|LGA| 214|      71|  -10|
|     AA|ORD|1197|     195|  -11|
+-------+---+----+--------+-----+
only showing top 10 rows


### 3.3. Đọc dữ liệu SMS với schema tường minh

Định nghĩa `StructType`.

In [10]:
sms_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("text", StringType(), True),
    StructField("label", IntegerType(), True)
])

sms = spark.read.csv(
    str(DATA_DIR / "sms.csv"),
    sep=";",
    header=False,
    schema=sms_schema
)

sms.show(5, truncate=False)
sms.printSchema()

+---+---------------------------------------------------------------------------------------------------------------+-----+
|id |text                                                                                                           |label|
+---+---------------------------------------------------------------------------------------------------------------+-----+
|1  |Sorry, I'll call later in meeting                                                                              |0    |
|2  |Dont worry. I guess he's busy.                                                                                 |0    |
|3  |Call FREEPHONE 0800 542 0578 now!                                                                              |1    |
|4  |Win a 1000 cash prize or a prize worth 5000                                                                    |1    |
|5  |Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...|0    |
+---+---

<a id="prep"></a>
## 4. Chuẩn bị dữ liệu

Trong Machine Learning, dữ liệu thô thường chưa thể đưa trực tiếp vào mô hình.

Các bước thường gặp:

1. loại cột không cần thiết;
2. xử lý missing values;
3. chuyển đổi đơn vị;
4. tạo label;
5. mã hóa biến phân loại;
6. ghép các predictor thành `features`.

### 4.1. Kiểm tra missing values

In [11]:
flights.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in flights.columns
]).show()

+---+---+---+-------+------+---+----+------+--------+-----+
|mon|dom|dow|carrier|flight|org|mile|depart|duration|delay|
+---+---+---+-------+------+---+----+------+--------+-----+
|  0|  0|  0|      0|     0|  0|   0|     0|       0|16711|
+---+---+---+-------+------+---+----+------+--------+-----+



### 4.2. Loại bỏ dữ liệu thiếu

In [12]:
flights_clean = (
    flights
    .drop("flight")
    .filter(F.col("delay").isNotNull())
    .dropna()
)

print("Trước:", flights.count())
print("Sau  :", flights_clean.count())

Trước: 275000
Sau  : 258289


### 4.3. Biến đổi cột

`withColumn()` được dùng để tạo feature mới.

Ở đây:

- mile → km;
- tạo `label = 1` nếu delay ≥ 15 phút.

In [13]:
flights_ml = (
    flights_clean
    .withColumn("km", F.round(F.col("mile") * 1.60934, 0))
    .drop("mile")
    .withColumn("label", (F.col("delay") >= 15).cast("integer"))
)

flights_ml.select(
    "mon", "carrier", "org", "km", "duration", "delay", "label"
).show(10)

+---+-------+---+------+--------+-----+-----+
|mon|carrier|org|    km|duration|delay|label|
+---+-------+---+------+--------+-----+-----+
| 10|     OO|ORD| 253.0|      51|   27|    1|
| 11|     OO|ORD|1188.0|     127|  -19|    0|
|  2|     B6|JFK|3618.0|     365|   60|    1|
|  5|     WN|SJC| 621.0|      85|   22|    1|
|  3|     B6|LGA|1732.0|     182|   70|    1|
|  5|     B6|ORD|1191.0|     130|   47|    1|
|  1|     UA|SFO|1093.0|     123|  135|    1|
|  8|     US|LGA| 344.0|      71|  -10|    0|
|  5|     AA|ORD|1926.0|     195|  -11|    0|
|  8|     B6|JFK|1902.0|     198|   20|    1|
+---+-------+---+------+--------+-----+-----+
only showing top 10 rows


<a id="categorical"></a>
## 5. Mã hóa dữ liệu phân loại

Machine Learning algorithms thường cần dữ liệu số.

### 5.1. StringIndexer

`StringIndexer` biến chuỗi category thành chỉ số số.

Ví dụ:

```text
AA -> 0
OO -> 1
B6 -> 2
...
```

Thứ tự mặc định thường phụ thuộc tần suất xuất hiện.

In [14]:
from pyspark.ml.feature import StringIndexer

carrier_indexer = StringIndexer(
    inputCol="carrier",
    outputCol="carrier_idx",
    handleInvalid="keep"
)

carrier_model = carrier_indexer.fit(flights_ml)
flights_idx = carrier_model.transform(flights_ml)

org_indexer = StringIndexer(
    inputCol="org",
    outputCol="org_idx",
    handleInvalid="keep"
)

org_model = org_indexer.fit(flights_idx)
flights_idx = org_model.transform(flights_idx)

flights_idx.select(
    "carrier", "carrier_idx", "org", "org_idx"
).show(10)

+-------+-----------+---+-------+
|carrier|carrier_idx|org|org_idx|
+-------+-----------+---+-------+
|     OO|        2.0|ORD|    0.0|
|     OO|        2.0|ORD|    0.0|
|     B6|        4.0|JFK|    2.0|
|     WN|        3.0|SJC|    5.0|
|     B6|        4.0|LGA|    3.0|
|     B6|        4.0|ORD|    0.0|
|     UA|        0.0|SFO|    1.0|
|     US|        6.0|LGA|    3.0|
|     AA|        1.0|ORD|    0.0|
|     B6|        4.0|JFK|    2.0|
+-------+-----------+---+-------+
only showing top 10 rows


### 5.2. Lưu ý về label

Trong slide, `StringIndexer` cũng được dùng để tạo `label` từ category.  
Ở dataset chuyến bay, label đã được tạo trực tiếp từ điều kiện:

```python
delay >= 15
```

Nghĩa là:

- `0`: không trễ đáng kể;
- `1`: trễ từ 15 phút trở lên.

<a id="assembler"></a>
## 6. VectorAssembler và cột features

Spark ML yêu cầu predictor được gom trong một cột vector tên thường là **`features`**.

`VectorAssembler` thực hiện việc này.

In [15]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=[
        "mon", "dom", "dow",
        "carrier_idx", "org_idx",
        "km", "depart", "duration"
    ],
    outputCol="features"
)

flights_ready = assembler.transform(flights_idx)

flights_ready.select(
    "features", "label", "delay"
).show(5, truncate=False)

+-----------------------------------------+-----+-----+
|features                                 |label|delay|
+-----------------------------------------+-----+-----+
|[10.0,10.0,1.0,2.0,0.0,253.0,8.18,51.0]  |1    |27   |
|[11.0,22.0,1.0,2.0,0.0,1188.0,7.17,127.0]|0    |-19  |
|[2.0,14.0,5.0,4.0,2.0,3618.0,21.17,365.0]|1    |60   |
|[5.0,25.0,3.0,3.0,5.0,621.0,12.92,85.0]  |1    |22   |
|[3.0,28.0,1.0,4.0,3.0,1732.0,13.33,182.0]|1    |70   |
+-----------------------------------------+-----+-----+
only showing top 5 rows


### 6.1. Chia train/test

In [16]:
train_df, test_df = flights_ready.randomSplit(
    [0.8, 0.2],
    seed=23
)

print("Train:", train_df.count())
print("Test :", test_df.count())
print("Train ratio:", train_df.count() / flights_ready.count())

Train: 206212
Test : 52077
Train ratio: 0.7983770117968632


<a id="dt"></a>
## 7. Classification: Decision Tree

Decision Tree phân chia dữ liệu qua các nút:

```text
Root
 ├── nhánh trái
 └── nhánh phải
```

Quá trình tiếp tục cho đến khi tạo các leaf node.

Decision Tree được dùng để phân loại xe theo nơi sản xuất.  
Ở phần mở rộng, ta dùng nó để dự đoán chuyến bay có trễ ≥ 15 phút hay không.

In [17]:
from pyspark.ml.classification import DecisionTreeClassifier

tree = DecisionTreeClassifier(
    labelCol="label",
    featuresCol="features",
    maxDepth=5,
    seed=23
)

tree_model = tree.fit(train_df)
tree_pred = tree_model.transform(test_df)

tree_pred.select(
    "label", "prediction", "probability"
).show(10, truncate=False)

+-----+----------+----------------------------------------+
|label|prediction|probability                             |
+-----+----------+----------------------------------------+
|1    |0.0       |[0.6225186017127615,0.37748139828723853]|
|1    |1.0       |[0.3195589266923886,0.6804410733076114] |
|0    |1.0       |[0.3195589266923886,0.6804410733076114] |
|1    |0.0       |[0.6225186017127615,0.37748139828723853]|
|1    |1.0       |[0.3195589266923886,0.6804410733076114] |
|0    |1.0       |[0.3195589266923886,0.6804410733076114] |
|1    |1.0       |[0.3195589266923886,0.6804410733076114] |
|1    |0.0       |[0.6225186017127615,0.37748139828723853]|
|0    |0.0       |[0.6225186017127615,0.37748139828723853]|
|0    |0.0       |[0.6225186017127615,0.37748139828723853]|
+-----+----------+----------------------------------------+
only showing top 10 rows


### 7.1. Xem cấu trúc cây

In [18]:
print(tree_model.toDebugString[:5000])

DecisionTreeClassificationModel: uid=DecisionTreeClassifier_4bd958cc807e, depth=5, numNodes=43, numClasses=2, numFeatures=8
  If (feature 6 <= 10.96)
   If (feature 4 in {4.0,5.0,6.0,7.0})
    If (feature 7 <= 251.5)
     Predict: 0.0
    Else (feature 7 > 251.5)
     If (feature 3 in {1.0,4.0,6.0})
      If (feature 4 in {5.0})
       Predict: 0.0
      Else (feature 4 not in {5.0})
       Predict: 1.0
     Else (feature 3 not in {1.0,4.0,6.0})
      If (feature 0 <= 8.5)
       Predict: 0.0
      Else (feature 0 > 8.5)
       Predict: 1.0
   Else (feature 4 not in {4.0,5.0,6.0,7.0})
    If (feature 6 <= 9.49)
     Predict: 0.0
    Else (feature 6 > 9.49)
     If (feature 0 <= 5.5)
      If (feature 5 <= 1050.5)
       Predict: 0.0
      Else (feature 5 > 1050.5)
       Predict: 1.0
     Else (feature 0 > 5.5)
      If (feature 0 <= 10.5)
       Predict: 0.0
      Else (feature 0 > 10.5)
       Predict: 1.0
  Else (feature 6 > 10.96)
   If (feature 4 in {4.0,5.0,6.0,7.0})
    If (feat

<a id="logistic"></a>
## 8. Classification: Logistic Regression

Logistic Regression dùng hàm logistic để biến linear score thành xác suất trong khoảng 0–1.

Mặc định:

- xác suất ≥ ngưỡng → class 1;
- xác suất < ngưỡng → class 0.

PySpark:

In [19]:
from pyspark.ml.classification import LogisticRegression

logistic = LogisticRegression(
    labelCol="label",
    featuresCol="features",
    maxIter=50
)

log_model = logistic.fit(train_df)
log_pred = log_model.transform(test_df)

log_pred.select(
    "label", "prediction", "probability"
).show(10, truncate=False)

+-----+----------+----------------------------------------+
|label|prediction|probability                             |
+-----+----------+----------------------------------------+
|1    |0.0       |[0.5572457916447638,0.44275420835523616]|
|1    |1.0       |[0.264630855404031,0.735369144595969]   |
|0    |1.0       |[0.24385733281275046,0.7561426671872495]|
|1    |0.0       |[0.5405863866128274,0.45941361338717257]|
|1    |1.0       |[0.29158128138704725,0.7084187186129527]|
|0    |1.0       |[0.3137372708473037,0.6862627291526964] |
|1    |1.0       |[0.31804904303673537,0.6819509569632647]|
|1    |1.0       |[0.424192724035195,0.575807275964805]   |
|0    |0.0       |[0.5356616493670753,0.4643383506329247] |
|0    |1.0       |[0.49707685136788565,0.5029231486321144]|
+-----+----------+----------------------------------------+
only showing top 10 rows


<a id="class_eval"></a>
## 9. Đánh giá mô hình phân loại

### 9.1. Confusion Matrix

Ta cần bốn thành phần:

- TP: dự đoán 1, thực tế 1.
- FP: dự đoán 1, thực tế 0.
- FN: dự đoán 0, thực tế 1.
- TN: dự đoán 0, thực tế 0.

In [20]:
tree_pred.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    0|       0.0|16543|
|    0|       1.0| 9061|
|    1|       0.0| 9919|
|    1|       1.0|16554|
+-----+----------+-----+



### 9.2. Tính Accuracy, Precision, Recall, F1

In [21]:
def binary_metrics(prediction_df):
    TP = prediction_df.filter("prediction = 1 AND label = 1").count()
    TN = prediction_df.filter("prediction = 0 AND label = 0").count()
    FP = prediction_df.filter("prediction = 1 AND label = 0").count()
    FN = prediction_df.filter("prediction = 0 AND label = 1").count()

    accuracy = (TP + TN) / (TP + TN + FP + FN)
    precision = TP / (TP + FP) if (TP + FP) else 0
    recall = TP / (TP + FN) if (TP + FN) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

    return {
        "TP": TP, "TN": TN, "FP": FP, "FN": FN,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

binary_metrics(tree_pred)

{'TP': 16554,
 'TN': 16543,
 'FP': 9061,
 'FN': 9919,
 'accuracy': 0.6355396816252856,
 'precision': 0.6462619558852235,
 'recall': 0.6253163600649718,
 'f1': 0.6356166487482722}

### 9.3. Evaluator của Spark

In [22]:
from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator,
    BinaryClassificationEvaluator
)

multi_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction"
)

binary_eval = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction"
)

for metric in ["accuracy", "f1", "weightedPrecision", "weightedRecall"]:
    value = multi_eval.evaluate(
        log_pred,
        {multi_eval.metricName: metric}
    )
    print(f"{metric:18s}: {value:.4f}")

auc = binary_eval.evaluate(log_pred)
print(f"{'AUC':18s}: {auc:.4f}")

accuracy          : 0.6059
f1                : 0.6055
weightedPrecision : 0.6058
weightedRecall    : 0.6059
AUC               : 0.6456


### 9.4. So sánh Decision Tree và Logistic Regression

In [23]:
def evaluate_classifier(name, pred):
    return {
        "model": name,
        "accuracy": multi_eval.evaluate(pred, {multi_eval.metricName: "accuracy"}),
        "f1": multi_eval.evaluate(pred, {multi_eval.metricName: "f1"}),
        "auc": binary_eval.evaluate(pred)
    }

results_classification = [
    evaluate_classifier("Decision Tree", tree_pred),
    evaluate_classifier("Logistic Regression", log_pred)
]

import pandas as pd
pd.DataFrame(results_classification)

,model,accuracy,f1,auc
0,Decision Tree,0.635540,0.635541,0.616441
1,Logistic Regression,0.605872,0.605523,0.645647


<a id="text"></a>
## 10. Text Mining: biến văn bản thành features

Slide trình bày pipeline:

```text
Text
 ↓
Remove punctuation
 ↓
Tokenizer
 ↓
StopWordsRemover
 ↓
HashingTF
 ↓
IDF
 ↓
features
```

Đây là cách biến văn bản thành vector số cho Machine Learning.

### 10.1. Làm sạch dấu câu và số

In [22]:
from pyspark.sql.functions import regexp_replace

sms_clean = sms.withColumn(
    "text",
    regexp_replace("text", r"[_():;,.!?\-]", " ")
)

sms_clean = sms_clean.withColumn(
    "text",
    regexp_replace("text", r"[0-9]", " ")
)

sms_clean = sms_clean.withColumn(
    "text",
    regexp_replace("text", r" +", " ")
)

sms_clean.show(5, truncate=False)

+---+-------------------------------------------------------------------------------------------------------+-----+
|id |text                                                                                                   |label|
+---+-------------------------------------------------------------------------------------------------------+-----+
|1  |Sorry I'll call later in meeting                                                                       |0    |
|2  |Dont worry I guess he's busy                                                                           |0    |
|3  |Call FREEPHONE now                                                                                     |1    |
|4  |Win a cash prize or a prize worth                                                                      |1    |
|5  |Go until jurong point crazy Available only in bugis n great world la e buffet Cine there got amore wat |0    |
+---+-------------------------------------------------------------------

### 10.2. Tokenizer

In [23]:
from pyspark.ml.feature import Tokenizer

tokenizer = Tokenizer(
    inputCol="text",
    outputCol="tokens"
)

sms_tokens = tokenizer.transform(sms_clean)

sms_tokens.select("text", "tokens").show(5, truncate=False)

+-------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------+
|text                                                                                                   |tokens                                                                                                                     |
+-------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------+
|Sorry I'll call later in meeting                                                                       |[sorry, i'll, call, later, in, meeting]                                                                                    |
|Dont worry I guess he's busy                                                   

### 10.3. StopWordsRemover

In [24]:
from pyspark.ml.feature import StopWordsRemover

remover = StopWordsRemover(
    inputCol="tokens",
    outputCol="words"
)

sms_words = remover.transform(sms_tokens)

sms_words.select("tokens", "words").show(5, truncate=False)

+---------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------+
|tokens                                                                                                                     |words                                                                                              |
+---------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------+
|[sorry, i'll, call, later, in, meeting]                                                                                    |[sorry, call, later, meeting]                                                                      |
|[dont, worry, i, guess, he's, busy]                                                            

### 10.4. HashingTF

In [25]:
from pyspark.ml.feature import HashingTF

hasher = HashingTF(
    inputCol="words",
    outputCol="hash",
    numFeatures=1024
)

sms_hash = hasher.transform(sms_words)

sms_hash.select("id", "words", "hash").show(5, truncate=False)

+---+---------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------+
|id |words                                                                                              |hash                                                                                                                                     |
+---+---------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------+
|1  |[sorry, call, later, meeting]                                                                      |(1024,[138,384,577,996],[1.0,1.0,1.0,1.0])                                                                                               |
|2  |[dont, worry, guess

### 10.5. IDF

In [26]:
from pyspark.ml.feature import IDF

idf = IDF(
    inputCol="hash",
    outputCol="features"
)

idf_model = idf.fit(sms_hash)
sms_tfidf = idf_model.transform(sms_hash)

sms_tfidf.select(
    "id", "label", "features"
).show(5, truncate=False)

+---+-----+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|id |label|features                                                                                                                                                                                                                                                                                                                                                                        |
+---+-----+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

<a id="sms"></a>
## 11. Classification văn bản SMS

Bài toán:

```text
Input : SMS text
Output: 0 = normal, 1 = spam
```

Ta dùng Logistic Regression trên TF-IDF features.

In [27]:
sms_model_df = sms_tfidf.select("label", "features")

sms_train, sms_test = sms_model_df.randomSplit(
    [0.8, 0.2],
    seed=13
)

sms_logistic = LogisticRegression(
    labelCol="label",
    featuresCol="features",
    regParam=0.2
)

sms_log_model = sms_logistic.fit(sms_train)
sms_pred = sms_log_model.transform(sms_test)

sms_pred.groupBy("label", "prediction").count().show()

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    1|       0.0|   39|
|    0|       0.0|  932|
|    1|       1.0|  121|
|    0|       1.0|    4|
+-----+----------+-----+



In [28]:
sms_auc = BinaryClassificationEvaluator(
    labelCol="label"
).evaluate(sms_pred)

sms_f1 = MulticlassClassificationEvaluator(
    labelCol="label",
    metricName="f1"
).evaluate(sms_pred)

print("SMS AUC:", sms_auc)
print("SMS F1 :", sms_f1)

SMS AUC: 0.9912793803418793
SMS F1 : 0.9587173795339076


<a id="onehot"></a>
## 12. One-Hot Encoding và Sparse Vector

### 12.1. Vấn đề với StringIndexer

Nếu một category được mã hóa:

```text
A -> 0
B -> 1
C -> 2
```

một số mô hình có thể hiểu sai rằng `C > B > A`.

One-Hot Encoding tạo vector binary cho category.

In [37]:
from pyspark.ml.feature import OneHotEncoder

onehot = OneHotEncoder(
    inputCols=["org_idx"],
    outputCols=["org_dummy"],
    dropLast=True
)

onehot_model = onehot.fit(flights_idx)
flights_ohe = onehot_model.transform(flights_idx)

flights_ohe.select(
    "org", "org_idx", "org_dummy"
).distinct().orderBy("org_idx").show(truncate=False)

+---+-------+-------------+
|org|org_idx|org_dummy    |
+---+-------+-------------+
|ORD|0.0    |(8,[0],[1.0])|
|SFO|1.0    |(8,[1],[1.0])|
|JFK|2.0    |(8,[2],[1.0])|
|LGA|3.0    |(8,[3],[1.0])|
|SMF|4.0    |(8,[4],[1.0])|
|SJC|5.0    |(8,[5],[1.0])|
|TUS|6.0    |(8,[6],[1.0])|
|OGG|7.0    |(8,[7],[1.0])|
+---+-------+-------------+



### 12.2. Dense và Sparse Vector

One-Hot Encoding thường tạo sparse vector vì phần lớn phần tử là 0.

In [38]:
from pyspark.ml.linalg import Vectors

dense = Vectors.dense([1, 0, 0, 0, 0, 7, 0, 0])
sparse = Vectors.sparse(8, [0, 5], [1, 7])

print("Dense :", dense)
print("Sparse:", sparse)

Dense : [1.0,0.0,0.0,0.0,0.0,7.0,0.0,0.0]
Sparse: (8,[0,5],[1.0,7.0])


<a id="regression"></a>
## 13. Regression với PySpark

Linear Regression để dự đoán fuel consumption.  
Phần mở rộng dùng dataset chuyến bay để dự đoán **duration**.

Ta thử mô hình đơn giản:

```text
duration ~ km
```

### 13.1. Tạo feature

In [39]:
reg_base = (
    flights
    .filter(F.col("duration").isNotNull())
    .filter(F.col("mile").isNotNull())
    .withColumn("km", F.round(F.col("mile") * 1.60934, 0))
)

reg_assembler = VectorAssembler(
    inputCols=["km"],
    outputCol="features"
)

reg_data = reg_assembler.transform(reg_base)

reg_train, reg_test = reg_data.randomSplit([0.8, 0.2], seed=42)

### 13.2. Linear Regression

In [40]:
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(
    labelCol="duration",
    featuresCol="features"
)

lr_model = lr.fit(reg_train)
reg_pred = lr_model.transform(reg_test)

reg_pred.select(
    "km", "duration", "prediction"
).show(10)

+------+--------+------------------+
|    km|duration|        prediction|
+------+--------+------------------+
|4001.0|     300| 347.3336415989867|
|3983.0|     375| 345.9700268069072|
|1395.0|     135|149.91252225681146|
|1926.0|     180|190.13915862315645|
|1180.0|     115| 133.6249011291953|
|1180.0|     130| 133.6249011291953|
|1180.0|     130| 133.6249011291953|
|1091.0|     110| 126.8825835461356|
|2356.0|     205|222.71440087838872|
|2659.0|     251|245.66858321172683|
+------+--------+------------------+
only showing top 10 rows


### 13.3. RMSE, MAE, MSE, R²

In [41]:
from pyspark.ml.evaluation import RegressionEvaluator

def regression_metric(pred, metric):
    return RegressionEvaluator(
        labelCol="duration",
        predictionCol="prediction",
        metricName=metric
    ).evaluate(pred)

for metric in ["rmse", "mae", "mse", "r2"]:
    print(metric, regression_metric(reg_pred, metric))

rmse 17.147609074128376
mae 13.112082394023453
mse 294.0404969591298
r2 0.9613277561376516


### 13.4. Hệ số và intercept

In [42]:
print("Intercept   :", lr_model.intercept)
print("Coefficients:", lr_model.coefficients)

Intercept   : 44.23237587065089
Coefficients: [0.07575637733774951]


### 13.5. Regression với categorical feature

Ta thêm sân bay xuất phát bằng One-Hot Encoding.

In [43]:
reg_idx = StringIndexer(
    inputCol="org",
    outputCol="org_idx",
    handleInvalid="keep"
).fit(reg_base).transform(reg_base)

reg_ohe = OneHotEncoder(
    inputCols=["org_idx"],
    outputCols=["org_dummy"]
).fit(reg_idx).transform(reg_idx)

reg_assembler2 = VectorAssembler(
    inputCols=["km", "org_dummy"],
    outputCol="features2"
)

reg_full = reg_assembler2.transform(reg_ohe)

reg_train2, reg_test2 = reg_full.randomSplit([0.8, 0.2], seed=42)

lr2 = LinearRegression(
    labelCol="duration",
    featuresCol="features2"
).fit(reg_train2)

pred2 = lr2.transform(reg_test2)

rmse2 = RegressionEvaluator(
    labelCol="duration",
    predictionCol="prediction",
    metricName="rmse"
).evaluate(pred2)

print("RMSE:", rmse2)

RMSE: 11.230385302359057


<a id="engineering"></a>
## 14. Feature Engineering và Bucketing

- bucketing;
- log, sqrt, power;
- product;
- ratio.

Bucketing biến biến liên tục thành nhóm rời rạc.

### 14.1. Bucketing giờ khởi hành

In [44]:
from pyspark.ml.feature import Bucketizer

# Dữ liệu depart có dạng giờ thập phân khoảng 0–24
bucketizer = Bucketizer(
    splits=[0, 6, 12, 18, 24],
    inputCol="depart",
    outputCol="depart_bucket",
    handleInvalid="keep"
)

bucketed = bucketizer.transform(
    flights.filter(F.col("depart").isNotNull())
)

bucketed.select(
    "depart", "depart_bucket"
).show(10)

bucketed.groupBy("depart_bucket").count().orderBy("depart_bucket").show()

+------+-------------+
|depart|depart_bucket|
+------+-------------+
|  8.18|          1.0|
|  15.5|          2.0|
|  7.17|          1.0|
| 21.17|          3.0|
| 12.92|          2.0|
| 13.33|          2.0|
|  9.58|          1.0|
| 12.75|          2.0|
|  13.0|          2.0|
| 14.42|          2.0|
+------+-------------+
only showing top 10 rows
+-------------+------+
|depart_bucket| count|
+-------------+------+
|          0.0|   911|
|          1.0| 98550|
|          2.0|104471|
|          3.0| 71068|
+-------------+------+



### 14.2. One-hot encode bucket

In [45]:
bucket_ohe = OneHotEncoder(
    inputCols=["depart_bucket"],
    outputCols=["depart_dummy"],
    dropLast=True
)

bucketed_ohe = bucket_ohe.fit(bucketed).transform(bucketed)

bucketed_ohe.select(
    "depart", "depart_bucket", "depart_dummy"
).show(10, truncate=False)

+------+-------------+-------------+
|depart|depart_bucket|depart_dummy |
+------+-------------+-------------+
|8.18  |1.0          |(3,[1],[1.0])|
|15.5  |2.0          |(3,[2],[1.0])|
|7.17  |1.0          |(3,[1],[1.0])|
|21.17 |3.0          |(3,[],[])    |
|12.92 |2.0          |(3,[2],[1.0])|
|13.33 |2.0          |(3,[2],[1.0])|
|9.58  |1.0          |(3,[1],[1.0])|
|12.75 |2.0          |(3,[2],[1.0])|
|13.0  |2.0          |(3,[2],[1.0])|
|14.42 |2.0          |(3,[2],[1.0])|
+------+-------------+-------------+
only showing top 10 rows


### 14.3. Feature engineering với công thức

In [46]:
engineered = (
    flights
    .filter(F.col("mile").isNotNull())
    .filter(F.col("duration").isNotNull())
    .withColumn("km", F.col("mile") * 1.60934)
    .withColumn("speed_km_per_min", F.col("km") / F.col("duration"))
    .withColumn("sqrt_km", F.sqrt("km"))
    .withColumn("log_km", F.log1p("km"))
)

engineered.select(
    "km", "duration", "speed_km_per_min", "sqrt_km", "log_km"
).show(10)

+----------+--------+-----------------+------------------+------------------+
|        km|duration| speed_km_per_min|           sqrt_km|            log_km|
+----------+--------+-----------------+------------------+------------------+
| 252.66638|      51|4.954242745098039| 15.89548300618764| 5.536019939100953|
| 749.95244|     102|7.352474901960784|27.385259538664226| 6.621342320863969|
|1187.69292|     127|9.351912755905511| 34.46292094411035|  7.08060959588236|
|3617.79632|     365|9.911770739726027| 60.14811983761421| 8.193896741239387|
| 621.20524|      85| 7.30829694117647|24.923989247309507|6.4332700061574215|
|1731.64984|     182| 9.51455956043956| 41.61309697679326| 7.457407215058698|
| 1190.9116|     130| 9.16085846153846|34.509587073739375| 7.083313683801401|
|1092.74186|     123|8.884080162601627| 33.05664623037249|6.9973599953584165|
| 344.39876|      71| 4.85068676056338|18.557983726687553| 5.844699575665601|
|1926.37998|     195|9.878871692307692| 43.89054545115611|  7.56

<a id="regularization"></a>
## 15. Regularization: Ridge, Lasso, Elastic Net

Linear Regression chuẩn tối thiểu hóa lỗi.  
Regularization bổ sung penalty lên coefficient.

### Ridge

- `elasticNetParam = 0`
- penalty L2.

### Lasso

- `elasticNetParam = 1`
- penalty L1.
- có thể đẩy một số coefficient về 0.

### Elastic Net

- `0 < elasticNetParam < 1`
- kết hợp L1 và L2.

In [47]:
reg_train3, reg_test3 = reg_full.randomSplit([0.8, 0.2], seed=23)

models_reg = []

for name, alpha in [
    ("Ridge", 0.0),
    ("ElasticNet", 0.5),
    ("Lasso", 1.0)
]:
    model = LinearRegression(
        labelCol="duration",
        featuresCol="features2",
        regParam=0.1,
        elasticNetParam=alpha
    ).fit(reg_train3)

    pred = model.transform(reg_test3)
    rmse = RegressionEvaluator(
        labelCol="duration",
        predictionCol="prediction",
        metricName="rmse"
    ).evaluate(pred)

    models_reg.append({
        "model": name,
        "rmse": rmse,
        "num_coefficients": len(model.coefficients),
        "zero_coefficients": sum(abs(x) < 1e-10 for x in model.coefficients)
    })

pd.DataFrame(models_reg)

,model,rmse,num_coefficients,zero_coefficients
0,Ridge,11.289664,9,0
1,ElasticNet,11.291365,9,1
2,Lasso,11.294537,9,0


<a id="pipeline"></a>
## 16. Pipeline và Data Leakage

```text
.fit()       -> chỉ dùng TRAIN
.transform() -> dùng cho TRAIN và TEST
```

Nếu fit bộ tiền xử lý trên toàn bộ dataset trước khi chia train/test, test data đã tác động vào preprocessing → **data leakage**.

`Pipeline` giúp đóng gói các stage đúng thứ tự.

### 16.1. Pipeline Regression

In [48]:
from pyspark.ml import Pipeline

pipeline_data = (
    flights
    .filter(F.col("duration").isNotNull())
    .filter(F.col("mile").isNotNull())
    .filter(F.col("org").isNotNull())
    .withColumn("km", F.round(F.col("mile") * 1.60934, 0))
)

pipe_train, pipe_test = pipeline_data.randomSplit(
    [0.8, 0.2],
    seed=13
)

pipe_indexer = StringIndexer(
    inputCol="org",
    outputCol="org_idx",
    handleInvalid="keep"
)

pipe_onehot = OneHotEncoder(
    inputCols=["org_idx"],
    outputCols=["org_dummy"]
)

pipe_assembler = VectorAssembler(
    inputCols=["km", "org_dummy"],
    outputCol="features"
)

pipe_lr = LinearRegression(
    labelCol="duration",
    featuresCol="features"
)

pipeline = Pipeline(
    stages=[
        pipe_indexer,
        pipe_onehot,
        pipe_assembler,
        pipe_lr
    ]
)

pipeline_model = pipeline.fit(pipe_train)
pipe_pred = pipeline_model.transform(pipe_test)

pipe_rmse = RegressionEvaluator(
    labelCol="duration",
    metricName="rmse"
).evaluate(pipe_pred)

print("Pipeline RMSE:", pipe_rmse)

Pipeline RMSE: 11.187971350851406


### 16.2. Xem các stages đã học

In [49]:
for i, stage in enumerate(pipeline_model.stages):
    print(i, type(stage).__name__)

0 StringIndexerModel
1 OneHotEncoderModel
2 VectorAssembler
3 LinearRegressionModel


<a id="cv"></a>
## 17. Cross Validation và Grid Search

Train/test split chỉ cho một phép chia dữ liệu.

Cross Validation:

1. chia training data thành nhiều fold;
2. lần lượt dùng mỗi fold làm validation;
3. tính metric trung bình;
4. chọn model/hyperparameter tốt nhất.

Test set vẫn được giữ riêng để đánh giá cuối cùng.

### 17.1. CrossValidator với parameter grid

In [50]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

cv_regression = LinearRegression(
    labelCol="duration",
    featuresCol="features"
)

cv_pipeline = Pipeline(
    stages=[
        pipe_indexer,
        pipe_onehot,
        pipe_assembler,
        cv_regression
    ]
)

param_grid = (
    ParamGridBuilder()
    .addGrid(cv_regression.regParam, [0.01, 0.1, 1.0])
    .addGrid(cv_regression.elasticNetParam, [0.0, 0.5, 1.0])
    .build()
)

print("Số cấu hình:", len(param_grid))

cv_evaluator = RegressionEvaluator(
    labelCol="duration",
    metricName="rmse"
)

cv = CrossValidator(
    estimator=cv_pipeline,
    estimatorParamMaps=param_grid,
    evaluator=cv_evaluator,
    numFolds=3,
    seed=13,
    parallelism=2
)

Số cấu hình: 9


> Cross Validation có thể mất thời gian. Với máy yếu, có thể giảm số cấu hình hoặc `numFolds`.

In [52]:
# Huấn luyện cross-validation
# Có thể mất vài phút tùy máy.
cv_model = cv.fit(pipe_train)

best_predictions = cv_model.transform(pipe_test)
best_rmse = cv_evaluator.evaluate(best_predictions)

print("Best test RMSE:", best_rmse)
print("Average CV metrics:")
print(cv_model.avgMetrics)

Best test RMSE: 11.18794629815799
Average CV metrics:
[np.float64(11.212223125266382), np.float64(11.212220771736682), np.float64(11.212250742825184), np.float64(11.212645439450162), np.float64(11.213404714714505), np.float64(11.215621374066805), np.float64(11.253953198119442), np.float64(11.329378009751911), np.float64(11.546458968429102)]


### 17.2. Xem hyperparameter tốt nhất

In [53]:
best_pipeline = cv_model.bestModel
best_lr = best_pipeline.stages[-1]

print("regParam       =", best_lr.getRegParam())
print("elasticNetParam=", best_lr.getElasticNetParam())

regParam       = 0.01
elasticNetParam= 0.5


<a id="ensemble"></a>
## 18. Ensemble Models

**Ensemble**: kết hợp nhiều cây/mô hình.

Các ví dụ trong code ZIP:

- Decision Tree;
- Gradient-Boosted Trees (GBT);
- Random Forest;
- Cross Validation cho Random Forest.

### 18.1. Chuẩn bị dữ liệu classification đơn giản

In [54]:
ensemble_data = (
    flights
    .filter(F.col("delay").isNotNull())
    .filter(F.col("duration").isNotNull())
    .filter(F.col("depart").isNotNull())
    .filter(F.col("mon").isNotNull())
    .withColumn("label", (F.col("delay") >= 15).cast("integer"))
)

ensemble_assembler = VectorAssembler(
    inputCols=["mon", "depart", "duration"],
    outputCol="features"
)

ensemble_data = ensemble_assembler.transform(ensemble_data)

ens_train, ens_test = ensemble_data.randomSplit(
    [0.8, 0.2],
    seed=17
)

### 18.2. Decision Tree, Random Forest, GBT

In [55]:
from pyspark.ml.classification import (
    DecisionTreeClassifier,
    RandomForestClassifier,
    GBTClassifier
)

models = {
    "Decision Tree": DecisionTreeClassifier(seed=17),
    "Random Forest": RandomForestClassifier(numTrees=30, seed=17),
    "GBT": GBTClassifier(maxIter=20, seed=17)
}

ensemble_results = []

for name, estimator in models.items():
    model = estimator.fit(ens_train)
    pred = model.transform(ens_test)

    auc = BinaryClassificationEvaluator(
        labelCol="label"
    ).evaluate(pred)

    f1 = MulticlassClassificationEvaluator(
        labelCol="label",
        metricName="f1"
    ).evaluate(pred)

    ensemble_results.append({
        "model": name,
        "AUC": auc,
        "F1": f1
    })

pd.DataFrame(ensemble_results).sort_values("AUC", ascending=False)

,model,AUC,F1
2,GBT,0.679001,0.628652
1,Random Forest,0.666685,0.620376
0,Decision Tree,0.618192,0.620846


### 18.3. Grid Search cho Random Forest

In [56]:
forest = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    seed=17
)

forest_grid = (
    ParamGridBuilder()
    .addGrid(forest.maxDepth, [3, 5, 8])
    .addGrid(forest.featureSubsetStrategy, ["sqrt", "log2"])
    .build()
)

forest_eval = BinaryClassificationEvaluator(
    labelCol="label",
    metricName="areaUnderROC"
)

forest_cv = CrossValidator(
    estimator=forest,
    estimatorParamMaps=forest_grid,
    evaluator=forest_eval,
    numFolds=3,
    seed=17,
    parallelism=2
)

# Có thể mất thời gian.
forest_cv_model = forest_cv.fit(ens_train)

print("Best test AUC:", forest_eval.evaluate(
    forest_cv_model.transform(ens_test)
))
print("Best maxDepth:", forest_cv_model.bestModel.getMaxDepth())
print("Best featureSubsetStrategy:",
      forest_cv_model.bestModel.getFeatureSubsetStrategy())

Best test AUC: 0.6778677001707498
Best maxDepth: 8
Best featureSubsetStrategy: sqrt


<a id="project"></a>
## BÀI TẬP MỞ RỘNG

### Bài toán 1 - Flight Delay Classification

#### Input

- month, day-of-month, day-of-week;
- carrier;
- origin airport;
- distance;
- departure time;
- duration.

#### Output

```text
label = 1 nếu delay >= 15 phút
label = 0 nếu ngược lại
```

#### Yêu cầu

1. Làm sạch dữ liệu.
2. Chia train/test.
3. Tạo preprocessing pipeline.
4. So sánh:
   - Logistic Regression;
   - Decision Tree;
   - Random Forest;
   - GBT.
5. Báo cáo Accuracy, F1, AUC.
6. Tuning ít nhất một mô hình bằng Cross Validation.
7. Giải thích vì sao chọn mô hình cuối cùng.

---

### Bài toán 2 - SMS Spam Classification

#### Input

Nội dung SMS.

#### Output

- 0: normal;
- 1: spam.

#### Pipeline

```text
Text
 → Tokenizer
 → StopWordsRemover
 → HashingTF
 → IDF
 → LogisticRegression
```

#### Yêu cầu mở rộng

- thử `numFeatures = 1024, 4096, 16384`;
- thử `regParam`;
- thử `elasticNetParam`;
- đánh giá AUC và F1.

<a id="exercise"></a>
## BÀI TẬP THÊM

### Bài 1 - Data Preparation

Với `flights-larger.csv`:

1. đếm missing value ở từng cột;
2. loại `flight`;
3. loại dòng thiếu `delay`;
4. tạo `km`;
5. tạo `label`.

### Import và chuẩn hóa tên cột

In [57]:
import math
import time

from pyspark.sql import functions as F
from pyspark.storagelevel import StorageLevel
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, VectorAssembler, Imputer
)
from pyspark.ml.classification import (
    LogisticRegression,
    DecisionTreeClassifier,
    RandomForestClassifier,
    GBTClassifier
)
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

SEED = 42

aliases = {
    "month": ["month", "mon"],
    "day_of_month": ["day_of_month", "day-of-month", "dom"],
    "day_of_week": ["day_of_week", "day-of-week", "dow"],
    "carrier": ["carrier"],
    "origin": ["origin", "origin_airport", "org"],
    "distance": ["distance", "mile"],
    "departure_time": ["departure_time", "depart"],
    "duration": ["duration"],
    "delay": ["delay"]
}

available = {c.strip().lower(): c for c in flights.columns}
selected = []

for target, candidates in aliases.items():
    source = next(
        (available[c] for c in candidates if c in available),
        None
    )
    if source is None:
        raise ValueError(
            f"Thiếu cột {target}. Các cột hiện có: {flights.columns}"
        )
    selected.append(F.col(source).alias(target))

raw = flights.select(*selected)
raw.printSchema()

root
 |-- month: integer (nullable = true)
 |-- day_of_month: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- carrier: string (nullable = true)
 |-- origin: string (nullable = true)
 |-- distance: integer (nullable = true)
 |-- departure_time: double (nullable = true)
 |-- duration: integer (nullable = true)
 |-- delay: integer (nullable = true)



### Làm sạch và tạo label

In [58]:
numeric_cols = [
    "month", "day_of_month", "day_of_week",
    "distance", "departure_time", "duration"
]
categorical_cols = ["carrier", "origin"]

# try_cast: giá trị không chuyển được thành số -> null
for c in numeric_cols + ["delay"]:
    raw = raw.withColumn(c, F.expr(f"try_cast(`{c}` AS DOUBLE)"))
    raw = raw.withColumn(
        c,
        F.when(
            F.isnan(F.col(c)) |
            (F.abs(F.col(c)) == F.lit(float("inf"))),
            F.lit(None).cast("double")
        ).otherwise(F.col(c))
    )

for c in categorical_cols:
    raw = raw.withColumn(
        c,
        F.when(
            F.col(c).isNull() |
            (F.trim(F.col(c)) == "") |
            F.upper(F.trim(F.col(c))).isin("NA", "N/A", "NULL"),
            F.lit("__MISSING__")
        ).otherwise(F.upper(F.trim(F.col(c))))
    )

# Giá trị không hợp lý chuyển thành null.
# dow chấp nhận cả quy ước 0–6 và 1–7; cần đối chiếu mô tả dữ liệu.
valid_rules = {
    "month": F.col("month").between(1, 12),
    "day_of_month": F.col("day_of_month").between(1, 31),
    "day_of_week": F.col("day_of_week").between(0, 7),
    "distance": F.col("distance") > 0,
    "duration": F.col("duration") > 0,
    "departure_time": F.col("departure_time") >= 0
}

for c, rule in valid_rules.items():
    raw = raw.withColumn(
        c,
        F.when(rule, F.col(c)).otherwise(F.lit(None).cast("double"))
    )

n_raw = raw.count()

data = (
    raw.filter(F.col("delay").isNotNull())
    .withColumn(
        "label",
        (F.col("delay") >= 15).cast("double")
    )
    # delay chỉ dùng tạo label, không đưa vào features
    .select(*numeric_cols, *categorical_cols, "label")
    .persist(StorageLevel.MEMORY_AND_DISK)
)

n_clean = data.count()

print("Số dòng ban đầu:", n_raw)
print("Số dòng dùng mô hình:", n_clean)
print("Loại do thiếu/không hợp lệ delay:", n_raw - n_clean)

data.groupBy("label").count().orderBy("label").show()

if data.select("label").distinct().count() != 2:
    raise ValueError("Dữ liệu cần có cả label 0 và label 1.")

Số dòng ban đầu: 275000
Số dòng dùng mô hình: 258289
Loại do thiếu/không hợp lệ delay: 16711
+-----+------+
|label| count|
+-----+------+
|  0.0|127701|
|  1.0|130588|
+-----+------+



### Chia train/validation/test

In [59]:
train, valid, test = data.randomSplit(
    [0.6, 0.2, 0.2],
    seed=SEED
)

for name, part in [("Train", train), ("Validation", valid), ("Test", test)]:
    part.persist(StorageLevel.MEMORY_AND_DISK)
    print(name, ":", part.count())
    part.groupBy("label").count().orderBy("label").show()

    if part.select("label").distinct().count() != 2:
        raise ValueError(f"{name} thiếu một lớp; cần điều chỉnh cách chia.")

# Median không thể tính nếu một cột hoàn toàn thiếu trên train
non_null = train.agg(*[
    F.count(c).alias(c) for c in numeric_cols
]).first()

empty_cols = [c for c in numeric_cols if non_null[c] == 0]

if empty_cols:
    raise ValueError(f"Các cột hoàn toàn thiếu trên train: {empty_cols}")

Train : 155314
+-----+-----+
|label|count|
+-----+-----+
|  0.0|76693|
|  1.0|78621|
+-----+-----+

Validation : 51249
+-----+-----+
|label|count|
+-----+-----+
|  0.0|25481|
|  1.0|25768|
+-----+-----+

Test : 51726
+-----+-----+
|label|count|
+-----+-----+
|  0.0|25527|
|  1.0|26199|
+-----+-----+



### Tạo preprocessing pipeline

In [60]:
def make_pipeline(classifier):
    imputer = Imputer(
        inputCols=numeric_cols,
        outputCols=[f"{c}_filled" for c in numeric_cols],
        strategy="median"
    )

    indexers = [
        StringIndexer(
            inputCol=c,
            outputCol=f"{c}_idx",
            handleInvalid="keep"
        )
        for c in categorical_cols
    ]

    encoder = OneHotEncoder(
        inputCols=[f"{c}_idx" for c in categorical_cols],
        outputCols=[f"{c}_vec" for c in categorical_cols],
        handleInvalid="keep"
    )

    assembler = VectorAssembler(
        inputCols=(
            [f"{c}_filled" for c in numeric_cols] +
            [f"{c}_vec" for c in categorical_cols]
        ),
        outputCol="features"
    )

    return Pipeline(
        stages=[imputer] + indexers + [encoder, assembler, classifier]
    )

### So sánh bốn mô hình

In [61]:
auc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

def evaluate(model, dataset):
    predictions = model.transform(dataset).persist(
        StorageLevel.MEMORY_AND_DISK
    )

    try:
        accuracy = MulticlassClassificationEvaluator(
            labelCol="label",
            predictionCol="prediction",
            metricName="accuracy"
        ).evaluate(predictions)

        weighted_f1 = MulticlassClassificationEvaluator(
            labelCol="label",
            predictionCol="prediction",
            metricName="f1"
        ).evaluate(predictions)

        auc = auc_evaluator.evaluate(predictions)

        counts = {
            (int(r["label"]), int(r["prediction"])): r["count"]
            for r in predictions.groupBy("label", "prediction").count().collect()
        }

        tp = counts.get((1, 1), 0)
        fp = counts.get((0, 1), 0)
        fn = counts.get((1, 0), 0)

        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1_delay = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )

        return {
            "accuracy": accuracy,
            "f1_weighted": weighted_f1,
            "auc": auc,
            "precision_delay": precision,
            "recall_delay": recall,
            "f1_delay": f1_delay
        }
    finally:
        predictions.unpersist()


classifiers = {
    "Logistic Regression": LogisticRegression(
        maxIter=50, regParam=0.01
    ),
    "Decision Tree": DecisionTreeClassifier(
        maxDepth=5, seed=SEED
    ),
    "Random Forest": RandomForestClassifier(
        numTrees=30, maxDepth=5, seed=SEED
    ),
    "GBT": GBTClassifier(
        maxIter=20, maxDepth=4, seed=SEED
    )
}

candidates = {}
results = []

for name, classifier in classifiers.items():
    print(f"Đang huấn luyện {name}...", flush=True)
    start = time.perf_counter()

    pipeline = make_pipeline(classifier)
    fitted = pipeline.fit(train)
    fit_seconds = time.perf_counter() - start

    metrics = evaluate(fitted, valid)

    candidates[name] = {
        "pipeline": pipeline,
        "model": fitted
    }

    results.append({
        "model": name,
        **metrics,
        "fit_seconds": fit_seconds
    })

    print(name, metrics, flush=True)

Đang huấn luyện Logistic Regression...
Logistic Regression {'accuracy': 0.6128900076099045, 'f1_weighted': 0.6122489428098585, 'auc': 0.6564297886923214, 'precision_delay': 0.6069714574387471, 'recall_delay': 0.6527864017385905, 'f1_delay': 0.6290458293599596}
Đang huấn luyện Decision Tree...
Decision Tree {'accuracy': 0.625339031005483, 'f1_weighted': 0.6245905794518658, 'auc': 0.5902006874843809, 'precision_delay': 0.6176165060715693, 'recall_delay': 0.6691244954982924, 'f1_delay': 0.6423395734376455}
Đang huấn luyện Random Forest...
Random Forest {'accuracy': 0.6264317352533708, 'f1_weighted': 0.6253441553618055, 'auc': 0.6789065647966955, 'precision_delay': 0.6166472929655853, 'recall_delay': 0.6793697609438063, 'f1_delay': 0.646490758350721}
Đang huấn luyện GBT...
GBT {'accuracy': 0.654607894788191, 'f1_weighted': 0.6544820437823913, 'auc': 0.7063097626929532, 'precision_delay': 0.651561267049938, 'recall_delay': 0.6729276622167029, 'f1_delay': 0.6620721253889769}


### Tuning Logistic Regression bằng Cross Validation

In [62]:
lr_tune = LogisticRegression(maxIter=50)
lr_pipeline = make_pipeline(lr_tune)

param_grid = (
    ParamGridBuilder()
    .addGrid(lr_tune.regParam, [0.01, 0.1])
    .addGrid(lr_tune.elasticNetParam, [0.0, 1.0])
    .build()
)

cv = CrossValidator(
    estimator=lr_pipeline,
    estimatorParamMaps=param_grid,
    evaluator=auc_evaluator,
    numFolds=3,
    seed=SEED,
    parallelism=1,
    collectSubModels=False
)

print("Đang chạy Cross Validation...", flush=True)
start = time.perf_counter()

cv_model = cv.fit(train)

cv_seconds = time.perf_counter() - start
best_lr_model = cv_model.bestModel
best_lr_stage = best_lr_model.stages[-1]

print("regParam tốt nhất:", best_lr_stage.getRegParam())
print("elasticNetParam tốt nhất:", best_lr_stage.getElasticNetParam())
print("AUC CV trung bình tốt nhất:", max(cv_model.avgMetrics))

# Pipeline mới với đúng cấu hình đã chọn, để refit sau này
tuned_pipeline = make_pipeline(
    LogisticRegression(
        maxIter=50,
        regParam=best_lr_stage.getRegParam(),
        elasticNetParam=best_lr_stage.getElasticNetParam()
    )
)

tuned_name = "Logistic Regression tuned"

candidates[tuned_name] = {
    "pipeline": tuned_pipeline,
    "model": best_lr_model
}

results.append({
    "model": tuned_name,
    **evaluate(best_lr_model, valid),
    "fit_seconds": cv_seconds
})

Đang chạy Cross Validation...
regParam tốt nhất: 0.01
elasticNetParam tốt nhất: 0.0
AUC CV trung bình tốt nhất: 0.6528359812426162


### Chọn mô hình và đánh giá trên test

In [63]:
print(
    f"{'Model':<28} {'Accuracy':>9} {'F1-w':>9} "
    f"{'AUC':>9} {'F1-delay':>10} {'Recall-1':>10} {'Fit(s)':>10}"
)

for r in sorted(results, key=lambda x: x["auc"], reverse=True):
    print(
        f"{r['model']:<28} "
        f"{r['accuracy']:>9.4f} "
        f"{r['f1_weighted']:>9.4f} "
        f"{r['auc']:>9.4f} "
        f"{r['f1_delay']:>10.4f} "
        f"{r['recall_delay']:>10.4f} "
        f"{r['fit_seconds']:>10.1f}"
    )

eligible = [r for r in results if math.isfinite(r["auc"])]

if not eligible:
    raise ValueError("Không có AUC hợp lệ để chọn mô hình.")

winner = max(
    eligible,
    key=lambda r: (r["auc"], r["f1_delay"])
)

best_name = winner["model"]

print("\nMô hình được chọn:", best_name)
print("AUC validation:", round(winner["auc"], 4))

# Refit toàn bộ pipeline trên train + validation
development = train.unionByName(valid)

final_model = candidates[best_name]["pipeline"].fit(development)

# Chỉ đánh giá mô hình đã chọn trên test
test_metrics = evaluate(final_model, test)

print("\nKẾT QUẢ TEST")
for metric, value in test_metrics.items():
    print(f"{metric}: {value:.4f}")

print("\nCONFUSION MATRIX")
(
    final_model.transform(test)
    .groupBy("label", "prediction")
    .count()
    .orderBy("label", "prediction")
    .show()
)

# Accuracy của cách luôn dự đoán lớp đông nhất
class_counts = [r["count"] for r in test.groupBy("label").count().collect()]
baseline_accuracy = max(class_counts) / sum(class_counts)

print(f"Majority-class baseline Accuracy: {baseline_accuracy:.4f}")

print(
    f"\nChọn {best_name} vì đạt AUC cao nhất trên validation "
    f"({winner['auc']:.4f}); F1 lớp delay là {winner['f1_delay']:.4f}. "
    f"Sau khi refit, AUC test đạt {test_metrics['auc']:.4f}, "
    f"Accuracy đạt {test_metrics['accuracy']:.4f} và "
    f"F1 lớp delay đạt {test_metrics['f1_delay']:.4f}."
)

for df in [data, train, valid, test]:
    df.unpersist()

Model                         Accuracy      F1-w       AUC   F1-delay   Recall-1     Fit(s)
GBT                             0.6546    0.6545    0.7063     0.6621     0.6729        7.1
Random Forest                   0.6264    0.6253    0.6789     0.6465     0.6794        3.7
Logistic Regression             0.6129    0.6122    0.6564     0.6290     0.6528        3.1
Logistic Regression tuned       0.6129    0.6122    0.6564     0.6290     0.6528       31.0
Decision Tree                   0.6253    0.6246    0.5902     0.6423     0.6691        2.4

Mô hình được chọn: GBT
AUC validation: 0.7063

KẾT QUẢ TEST
accuracy: 0.6477
f1_weighted: 0.6476
auc: 0.7015
precision_delay: 0.6485
recall_delay: 0.6648
f1_delay: 0.6566

CONFUSION MATRIX
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0|16086|
|  0.0|       1.0| 9441|
|  1.0|       0.0| 8781|
|  1.0|       1.0|17418|
+-----+----------+-----+

Majority-class baseline Accuracy: 0.5065

Chọn GBT vì đạt

GBT được chọn vì có AUC validation cao nhất (0,7063) và F1 lớp chuyến bay trễ tốt nhất trong các mô hình so sánh. Trên tập test, mô hình đạt Accuracy 64,77%, AUC 0,7015 và F1-delay 0,6566, với kết quả gần validation, chưa cho thấy dấu hiệu overfitting rõ rệt.

Accuracy cao hơn baseline 14,12 điểm phần trăm. Tuy nhiên, mô hình chỉ phát hiện 66,48% chuyến bay trễ, còn bỏ sót khoảng 33,52%, nên khả năng dự báo vẫn cần cải thiện. Tuning Logistic Regression chưa giúp cải thiện các chỉ số ở mức làm tròn được báo cáo.

### Bài 2 - StringIndexer và OneHotEncoder

1. index `carrier`;
2. index `org`;
3. one-hot encode `org`;
4. giải thích tại sao không nên dùng index trực tiếp như một biến liên tục.

In [64]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder

# 1. Index carrier
carrier_indexer = StringIndexer(
    inputCol="carrier",
    outputCol="carrier_index",
    handleInvalid="keep"
)

# 2. Index org
org_indexer = StringIndexer(
    inputCol="org",
    outputCol="org_index",
    handleInvalid="keep"
)

# 3. One-hot encode org
org_encoder = OneHotEncoder(
    inputCols=["org_index"],
    outputCols=["org_encoded"]
)

pipeline = Pipeline(
    stages=[carrier_indexer, org_indexer, org_encoder]
)

encoding_model = pipeline.fit(flights)
flights_encoded = encoding_model.transform(flights)

flights_encoded.select(
    "carrier", "carrier_index",
    "org", "org_index", "org_encoded"
).show(10, truncate=False)

# Xem ánh xạ tên → index
for name, model in zip(
    ["carrier", "org"],
    encoding_model.stages[:2]
):
    print(f"\nÁnh xạ {name}:")
    for index, label in enumerate(model.labels):
        print(f"{label} → {index}")

+-------+-------------+---+---------+-------------+
|carrier|carrier_index|org|org_index|org_encoded  |
+-------+-------------+---+---------+-------------+
|OO     |2.0          |ORD|0.0      |(8,[0],[1.0])|
|OO     |2.0          |ORD|0.0      |(8,[0],[1.0])|
|OO     |2.0          |ORD|0.0      |(8,[0],[1.0])|
|B6     |4.0          |JFK|2.0      |(8,[2],[1.0])|
|WN     |3.0          |SJC|5.0      |(8,[5],[1.0])|
|B6     |4.0          |LGA|3.0      |(8,[3],[1.0])|
|B6     |4.0          |ORD|0.0      |(8,[0],[1.0])|
|UA     |0.0          |SFO|1.0      |(8,[1],[1.0])|
|US     |6.0          |LGA|3.0      |(8,[3],[1.0])|
|AA     |1.0          |ORD|0.0      |(8,[0],[1.0])|
+-------+-------------+---+---------+-------------+
only showing top 10 rows

Ánh xạ carrier:
UA → 0
AA → 1
OO → 2
WN → 3
B6 → 4
OH → 5
US → 6
HA → 7
AQ → 8

Ánh xạ org:
ORD → 0
SFO → 1
JFK → 2
LGA → 3
SMF → 4
SJC → 5
TUS → 6
OGG → 7


### Bài 3 - Decision Tree

1. huấn luyện cây với `maxDepth = 3, 5, 10`;
2. so sánh Accuracy/F1/AUC;
3. in `toDebugString`.

In [65]:
import time
from pyspark.ml.classification import DecisionTreeClassifier

depth_results = []
tree_models = {}

for depth in [3, 5, 10]:
    print(f"\nĐang huấn luyện maxDepth={depth}...", flush=True)

    classifier = DecisionTreeClassifier(
        labelCol="label",
        featuresCol="features",
        maxDepth=depth,
        seed=42
    )

    start = time.perf_counter()
    model = make_pipeline(classifier).fit(train)
    fit_seconds = time.perf_counter() - start

    metrics = evaluate(model, valid)
    tree = model.stages[-1]

    tree_models[depth] = model
    depth_results.append({
        "depth": depth,
        **metrics,
        "seconds": fit_seconds,
        "nodes": tree.numNodes
    })

    # In cấu trúc cây
    print(f"\n--- toDebugString: maxDepth={depth} ---")
    print(tree.toDebugString)

print(
    f"\n{'Depth':<8} {'Accuracy':>10} {'F1-weighted':>13} "
    f"{'AUC':>10} {'Nodes':>8} {'Fit(s)':>10}"
)

for r in depth_results:
    print(
        f"{r['depth']:<8} "
        f"{r['accuracy']:>10.4f} "
        f"{r['f1_weighted']:>13.4f} "
        f"{r['auc']:>10.4f} "
        f"{r['nodes']:>8} "
        f"{r['seconds']:>10.2f}"
    )

best = max(depth_results, key=lambda r: r["auc"])
print(f"\nCây có AUC validation cao nhất: maxDepth={best['depth']}")


Đang huấn luyện maxDepth=3...

--- toDebugString: maxDepth=3 ---
DecisionTreeClassificationModel: uid=DecisionTreeClassifier_13980f7d7c36, depth=3, numNodes=7, numClasses=2, numFeatures=25
  If (feature 4 <= 11.49)
   Predict: 0.0
  Else (feature 4 > 11.49)
   If (feature 0 <= 6.5)
    Predict: 1.0
   Else (feature 0 > 6.5)
    If (feature 0 <= 10.5)
     Predict: 0.0
    Else (feature 0 > 10.5)
     Predict: 1.0


Đang huấn luyện maxDepth=5...

--- toDebugString: maxDepth=5 ---
DecisionTreeClassificationModel: uid=DecisionTreeClassifier_eb96f8b652fc, depth=5, numNodes=43, numClasses=2, numFeatures=25
  If (feature 4 <= 11.49)
   If (feature 9 in {1.0})
    If (feature 4 <= 9.01)
     If (feature 0 <= 10.5)
      Predict: 0.0
     Else (feature 0 > 10.5)
      If (feature 1 <= 1.5)
       Predict: 1.0
      Else (feature 1 > 1.5)
       Predict: 0.0
    Else (feature 4 > 9.01)
     If (feature 17 in {1.0})
      If (feature 4 <= 10.475000000000001)
       Predict: 0.0
      Else (feat

### Bài 4 - Logistic Regression

1. huấn luyện Logistic Regression;
2. tính confusion matrix;
3. tính Precision, Recall, F1;
4. tính AUC.

In [66]:
from pyspark.sql import functions as F
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.storagelevel import StorageLevel

# 1. Huấn luyện Logistic Regression
lr = LogisticRegression(
    labelCol="label",
    featuresCol="features",
    maxIter=50,
    regParam=0.01,
    elasticNetParam=0.0,
    threshold=0.5
)

lr_model = make_pipeline(lr).fit(train)

predictions = lr_model.transform(valid).persist(
    StorageLevel.MEMORY_AND_DISK
)

try:
    # 2. Confusion matrix: dòng = thực tế, cột = dự đoán
    print("CONFUSION MATRIX")
    (
        predictions.groupBy("label")
        .pivot("prediction", [0.0, 1.0])
        .count()
        .fillna(0)
        .orderBy("label")
        .show()
    )

    counts = {
        (int(r["label"]), int(r["prediction"])): r["count"]
        for r in (
            predictions.groupBy("label", "prediction")
            .count()
            .collect()
        )
    }

    tn = counts.get((0, 0), 0)
    fp = counts.get((0, 1), 0)
    fn = counts.get((1, 0), 0)
    tp = counts.get((1, 1), 0)

    # 3. Precision, Recall, F1 cho label = 1
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )

    # 4. ROC-AUC từ rawPrediction, không dùng nhãn prediction
    auc = BinaryClassificationEvaluator(
        labelCol="label",
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC"
    ).evaluate(predictions)

    print(f"TN={tn:,}, FP={fp:,}, FN={fn:,}, TP={tp:,}")
    print(f"Precision (delay=1): {precision:.4f}")
    print(f"Recall    (delay=1): {recall:.4f}")
    print(f"F1        (delay=1): {f1:.4f}")
    print(f"ROC-AUC           : {auc:.4f}")

finally:
    predictions.unpersist()

CONFUSION MATRIX
+-----+-----+-----+
|label|  0.0|  1.0|
+-----+-----+-----+
|  0.0|14589|10892|
|  1.0| 8947|16821|
+-----+-----+-----+

TN=14,589, FP=10,892, FN=8,947, TP=16,821
Precision (delay=1): 0.6070
Recall    (delay=1): 0.6528
F1        (delay=1): 0.6290
ROC-AUC           : 0.6564


### Bài 5 - SMS

1. loại punctuation;
2. tokenize;
3. remove stop words;
4. HashingTF;
5. IDF;
6. Logistic Regression;
7. tính F1 và AUC.

In [67]:
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    RegexTokenizer, StopWordsRemover, HashingTF, IDF
)
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.storagelevel import StorageLevel

# Làm sạch text và label
sms_clean = (
    sms.select(
        F.col("id"),
        F.col("text"),
        F.expr("try_cast(label AS DOUBLE)").alias("label")
    )
    .filter(
        F.col("text").isNotNull() &
        F.col("label").isin(0.0, 1.0)
    )
    # 1. Thay punctuation bằng khoảng trắng, chuyển chữ thường
    .withColumn(
        "clean_text",
        F.trim(
            F.regexp_replace(
                F.lower(F.col("text")),
                r"[\p{P}\p{S}]+",
                " "
            )
        )
    )
    .filter(F.length("clean_text") > 0)
)

# Giữ các SMS giống nhau trong cùng tập để tránh trùng text
# giữa train và test; chia theo hash của nội dung chuẩn hóa.
sms_split = sms_clean.withColumn(
    "_bucket",
    F.pmod(F.xxhash64("clean_text"), F.lit(100))
)

sms_train = sms_split.filter(F.col("_bucket") < 80).drop("_bucket")
sms_test = sms_split.filter(F.col("_bucket") >= 80).drop("_bucket")

for name, dataset in [("Train", sms_train), ("Test", sms_test)]:
    print(name)
    dataset.groupBy("label").count().orderBy("label").show()

    if dataset.select("label").distinct().count() != 2:
        raise ValueError(f"{name} cần có cả ham và spam.")

# 2. Tokenize
tokenizer = RegexTokenizer(
    inputCol="clean_text",
    outputCol="tokens",
    pattern=r"\s+",
    minTokenLength=1
)

# 3. Loại stop words tiếng Anh
stop_remover = StopWordsRemover(
    inputCol="tokens",
    outputCol="filtered_tokens",
    stopWords=StopWordsRemover.loadDefaultStopWords("english")
)

# 4. Term frequency
hashing_tf = HashingTF(
    inputCol="filtered_tokens",
    outputCol="tf",
    numFeatures=16384
)

# 5. TF-IDF
idf = IDF(
    inputCol="tf",
    outputCol="features"
)

# 6. Logistic Regression
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=50,
    regParam=0.1,
    elasticNetParam=0.0
)

sms_pipeline = Pipeline(
    stages=[tokenizer, stop_remover, hashing_tf, idf, lr]
)

print("Đang huấn luyện...", flush=True)
sms_model = sms_pipeline.fit(sms_train)

sms_predictions = sms_model.transform(sms_test).persist(
    StorageLevel.MEMORY_AND_DISK
)

try:
    sms_predictions.select(
        "text", "filtered_tokens", "label", "prediction"
    ).show(5, truncate=80)

    print("CONFUSION MATRIX")
    sms_predictions.groupBy("label", "prediction").count().orderBy(
        "label", "prediction"
    ).show()

    counts = {
        (int(r["label"]), int(r["prediction"])): r["count"]
        for r in sms_predictions.groupBy(
            "label", "prediction"
        ).count().collect()
    }

    tp = counts.get((1, 1), 0)
    fp = counts.get((0, 1), 0)
    fn = counts.get((1, 0), 0)

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0

    # 7. F1 riêng lớp spam và ROC-AUC
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )

    auc = BinaryClassificationEvaluator(
        labelCol="label",
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC"
    ).evaluate(sms_predictions)

    print(f"Precision spam: {precision:.4f}")
    print(f"Recall spam   : {recall:.4f}")
    print(f"F1 spam       : {f1:.4f}")
    print(f"ROC-AUC       : {auc:.4f}")

finally:
    sms_predictions.unpersist()

Train
+-----+-----+
|label|count|
+-----+-----+
|  0.0| 3894|
|  1.0|  590|
+-----+-----+

Test
+-----+-----+
|label|count|
+-----+-----+
|  0.0|  931|
|  1.0|  157|
+-----+-----+

Đang huấn luyện...
+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+-----+----------+
|                                                                            text|                                                                 filtered_tokens|label|prediction|
+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+-----+----------+
|                                               Sorry, I'll call later in meeting|                                               [sorry, ll, call, later, meeting]|  0.0|       0.0|
|Go until jurong point, crazy.. Available only in bugis n great world la e b

### Bài 6 - Regression

Dự đoán `duration`.

1. model chỉ dùng `km`;
2. model dùng `km + org`;
3. so sánh RMSE;
4. giải thích coefficient.

In [68]:
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, VectorAssembler
)
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# 1. Chuẩn bị dữ liệu
if "km" in flights.columns:
    km_expr = F.expr("try_cast(km AS DOUBLE)")
elif "mile" in flights.columns:
    km_expr = F.expr("try_cast(mile AS DOUBLE)") * 1.609344
else:
    raise ValueError("Dữ liệu cần có cột km hoặc mile.")

reg_data = (
    flights.select(
        km_expr.alias("km"),
        F.expr("try_cast(duration AS DOUBLE)").alias("duration"),
        F.upper(F.trim(F.col("org"))).alias("org")
    )
    .filter(
        F.col("km").isNotNull() &
        F.col("duration").isNotNull() &
        ~F.isnan("km") &
        ~F.isnan("duration") &
        (F.col("km") > 0) &
        (F.col("km") < float("inf")) &
        (F.col("duration") > 0) &
        (F.col("duration") < float("inf")) &
        F.col("org").isNotNull() &
        (F.col("org") != "")
    )
)

reg_train, reg_test = reg_data.randomSplit([0.8, 0.2], seed=42)

# 2. Model 1: chỉ dùng km
assembler_1 = VectorAssembler(
    inputCols=["km"],
    outputCol="features"
)

lr_1 = LinearRegression(
    featuresCol="features",
    labelCol="duration",
    maxIter=100,
    regParam=0.0
)

pipeline_1 = Pipeline(stages=[assembler_1, lr_1])

# 3. Model 2: km + org
org_indexer = StringIndexer(
    inputCol="org",
    outputCol="org_index",
    handleInvalid="keep"
)

org_encoder = OneHotEncoder(
    inputCols=["org_index"],
    outputCols=["org_vector"],
    dropLast=True
)

assembler_2 = VectorAssembler(
    inputCols=["km", "org_vector"],
    outputCol="features"
)

lr_2 = LinearRegression(
    featuresCol="features",
    labelCol="duration",
    maxIter=100,
    regParam=0.0
)

pipeline_2 = Pipeline(
    stages=[org_indexer, org_encoder, assembler_2, lr_2]
)

print("Đang huấn luyện...", flush=True)
reg_model_1 = pipeline_1.fit(reg_train)
reg_model_2 = pipeline_2.fit(reg_train)

# 4. So sánh RMSE trên cùng tập test
evaluator = RegressionEvaluator(
    labelCol="duration",
    predictionCol="prediction",
    metricName="rmse"
)

rmse_1 = evaluator.evaluate(reg_model_1.transform(reg_test))
rmse_2 = evaluator.evaluate(reg_model_2.transform(reg_test))

print(f"\n{'Model':<20} {'RMSE (phút)':>15}")
print(f"{'km':<20} {rmse_1:>15.4f}")
print(f"{'km + org':<20} {rmse_2:>15.4f}")

if rmse_1 > 0:
    improvement = (rmse_1 - rmse_2) / rmse_1 * 100
    print(f"Thay đổi RMSE khi thêm org: {improvement:.2f}%")
    print("(Dương = cải thiện; âm = giảm chất lượng)")

# 5. Coefficient của km
fitted_1 = reg_model_1.stages[-1]
fitted_2 = reg_model_2.stages[-1]

print("\nMODEL 1")
print(f"Intercept: {fitted_1.intercept:.4f}")
print(f"Coefficient km: {fitted_1.coefficients[0]:.6f}")
print(
    f"Tăng 100 km → duration dự đoán thay đổi "
    f"{100 * fitted_1.coefficients[0]:.2f} phút."
)

print("\nMODEL 2")
print(f"Intercept: {fitted_2.intercept:.4f}")
print(f"Coefficient km: {fitted_2.coefficients[0]:.6f}")
print(
    f"Giữ nguyên org, tăng 100 km → duration dự đoán thay đổi "
    f"{100 * fitted_2.coefficients[0]:.2f} phút."
)

# So sánh hiệu ứng org với một sân bay quan sát được.
# Lấy chênh lệch coefficient để dễ diễn giải.
labels = reg_model_2.stages[0].labels
coefficients = fitted_2.coefficients.toArray()

if labels:
    reference_org = labels[0]
    reference_effect = coefficients[1]

    print(f"\nHiệu ứng org so với {reference_org}, giữ nguyên km:")
    for i, airport in enumerate(labels):
        difference = coefficients[1 + i] - reference_effect
        print(f"{airport}: {difference:+.4f} phút")

Đang huấn luyện...

Model                    RMSE (phút)
km                           17.2637
km + org                     11.2119
Thay đổi RMSE khi thêm org: 35.06%
(Dương = cải thiện; âm = giảm chất lượng)

MODEL 1
Intercept: 44.2485
Coefficient km: 0.075730
Tăng 100 km → duration dự đoán thay đổi 7.57 phút.

MODEL 2
Intercept: 46.3295
Coefficient km: 0.074317
Giữ nguyên org, tăng 100 km → duration dự đoán thay đổi 7.43 phút.

Hiệu ứng org so với ORD, giữ nguyên km:
ORD: +0.0000 phút
SFO: -7.9204 phút
JFK: +24.4944 phút
LGA: +18.7050 phút
SMF: -12.5185 phút
SJC: -10.1977 phút
TUS: -10.3879 phút
OGG: -28.0666 phút


**Mô hình `km + org` dự đoán tốt hơn mô hình chỉ dùng `km`.** RMSE giảm từ **17,26 xuống 11,21 phút**, tương đương giảm **35,06%**. Điều này cho thấy sân bay xuất phát bổ sung thông tin hữu ích ngoài khoảng cách. RMSE phản ánh mức sai số tổng thể và nhạy với sai số lớn, không phải sai số tuyệt đối trung bình.

**Ảnh hưởng của khoảng cách:**

- Model 1: tăng **100 km**, thời lượng dự đoán tăng khoảng **7,57 phút**.
- Model 2: giữ nguyên sân bay xuất phát, tăng **100 km**, thời lượng dự đoán tăng khoảng **7,43 phút**.

Hai hệ số gần nhau cho thấy mối liên hệ giữa khoảng cách và thời lượng khá nhất quán trong hai mô hình.

**Ảnh hưởng của sân bay:** Với cùng khoảng cách, so với chuyến xuất phát từ **ORD**, mô hình dự đoán:

| Sân bay | Chênh lệch thời lượng |
|---|---:|
| JFK | Dài hơn 24,49 phút |
| LGA | Dài hơn 18,71 phút |
| SFO | Ngắn hơn 7,92 phút |
| OGG | Ngắn hơn 28,07 phút |

Các chênh lệch này là **mối liên hệ trong dữ liệu**, có thể phản ánh khác biệt về tuyến bay và điều kiện khai thác; chưa đủ để kết luận sân bay là nguyên nhân trực tiếp.

**Intercept** là giá trị dự đoán tại `km = 0` theo cách mã hóa của từng mô hình, nên ít ý nghĩa thực tế. Riêng `46,3295` của model 2 **không nên hiểu là thời lượng nền của ORD**, vì bảng hiệu ứng sân bay đã được quy đổi để so sánh với ORD.

Với kết quả trên cùng tập test, **chọn mô hình `km + org`** vì có RMSE thấp hơn.

### Bài 7 - Regularization

Huấn luyện:

- Linear Regression;
- Ridge;
- Lasso;
- Elastic Net.

So sánh:

- RMSE;
- số coefficient gần 0.

In [69]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, VectorAssembler
)
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# 1. Preprocessing: fit một lần trên train, dùng chung cho 4 model
preprocessing = Pipeline(stages=[
    StringIndexer(
        inputCol="org",
        outputCol="org_index",
        handleInvalid="keep"
    ),
    OneHotEncoder(
        inputCols=["org_index"],
        outputCols=["org_vector"],
        dropLast=True
    ),
    VectorAssembler(
        inputCols=["km", "org_vector"],
        outputCol="features"
    )
])

prep_model = preprocessing.fit(reg_train)

train_features = prep_model.transform(reg_train).select(
    "features", "duration"
).cache()

test_features = prep_model.transform(reg_test).select(
    "features", "duration"
).cache()

# 2. Cấu hình regularization
settings = [
    ("Linear Regression", 0.0, 0.0),
    ("Ridge",             0.1, 0.0),
    ("Lasso",             0.1, 1.0),
    ("Elastic Net",       0.1, 0.5)
]

evaluator = RegressionEvaluator(
    labelCol="duration",
    predictionCol="prediction",
    metricName="rmse"
)

# Quy ước coefficient gần 0: |coefficient| < 10^-6
EPSILON = 1e-6

regularization_models = {}
regularization_results = []

try:
    for name, reg_param, elastic_param in settings:
        print(f"Đang huấn luyện {name}...", flush=True)

        estimator = LinearRegression(
            labelCol="duration",
            featuresCol="features",
            regParam=reg_param,
            elasticNetParam=elastic_param,
            standardization=True,
            maxIter=200,
            tol=1e-6
        )

        model = estimator.fit(train_features)
        predictions = model.transform(test_features)

        rmse = evaluator.evaluate(predictions)

        coefficients = [float(x) for x in model.coefficients]
        near_zero = sum(abs(x) < EPSILON for x in coefficients)

        regularization_models[name] = model
        regularization_results.append({
            "model": name,
            "rmse": rmse,
            "near_zero": near_zero,
            "total": len(coefficients)
        })

    # 3. Bảng so sánh
    print(f"\nNgưỡng gần 0: |coefficient| < {EPSILON}")
    print(
        f"{'Model':<22} {'RMSE':>12} "
        f"{'Coef gần 0':>14} {'Tổng coef':>12}"
    )

    for result in regularization_results:
        print(
            f"{result['model']:<22} "
            f"{result['rmse']:>12.4f} "
            f"{result['near_zero']:>14} "
            f"{result['total']:>12}"
        )

finally:
    train_features.unpersist()
    test_features.unpersist()

Đang huấn luyện Linear Regression...
Đang huấn luyện Ridge...
Đang huấn luyện Lasso...
Đang huấn luyện Elastic Net...

Ngưỡng gần 0: |coefficient| < 1e-06
Model                          RMSE     Coef gần 0    Tổng coef
Linear Regression           11.2119              0            9
Ridge                       11.2119              0            9
Lasso                       11.2157              0            9
Elastic Net                 11.2131              1            9


**Linear Regression và Ridge cho RMSE thấp nhất, cùng đạt 11,2119 phút** ở mức làm tròn được báo cáo. Lasso và Elastic Net có RMSE cao hơn rất ít, nên regularization chưa cho thấy cải thiện khả năng dự báo tại mức phạt đang thử.

- **Ridge:** không có hệ số gần 0, phù hợp với cơ chế co nhỏ hệ số.
- **Lasso:** chưa đưa hệ số nào xuống dưới ngưỡng `10⁻⁶`; mức phạt hiện tại chưa tạo được mô hình thưa.
- **Elastic Net:** có **1/9 hệ số gần 0**, trong khi RMSE chỉ tăng nhẹ lên **11,2131 phút**.

Với kết quả hiện tại, có thể **giữ Linear Regression** vì đơn giản và đạt RMSE thấp nhất. Nếu ưu tiên giảm số đặc trưng, Elastic Net là phương án đáng cân nhắc, nhưng cần kiểm tra hệ số nào gần 0 và thử thêm các mức phạt bằng Cross Validation.

### Bài 8 - Pipeline

Xây dựng Pipeline:

```text
StringIndexer
 → OneHotEncoder
 → VectorAssembler
 → LinearRegression
```

Giải thích tại sao phải fit pipeline trên training data.

In [70]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, VectorAssembler
)
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# 1. Chuyển sân bay thành index
indexer = StringIndexer(
    inputCol="org",
    outputCol="org_index",
    handleInvalid="keep"
)

# 2. One-hot encode sân bay
encoder = OneHotEncoder(
    inputCols=["org_index"],
    outputCols=["org_vector"]
)

# 3. Gộp các đặc trưng
assembler = VectorAssembler(
    inputCols=["km", "org_vector"],
    outputCol="features"
)

# 4. Hồi quy tuyến tính
lr = LinearRegression(
    featuresCol="features",
    labelCol="duration",
    maxIter=200,
    regParam=0.0
)

pipeline = Pipeline(
    stages=[indexer, encoder, assembler, lr]
)

# Chỉ fit trên training data
pipeline_model = pipeline.fit(reg_train)

# Dùng pipeline đã học để dự đoán test
predictions = pipeline_model.transform(reg_test)

predictions.select(
    "km", "org", "duration", "prediction"
).show(10, truncate=False)

rmse = RegressionEvaluator(
    labelCol="duration",
    predictionCol="prediction",
    metricName="rmse"
).evaluate(predictions)

print(f"RMSE test: {rmse:.4f} phút")

+------------------+---+--------+-----------------+
|km                |org|duration|prediction       |
+------------------+---+--------+-----------------+
|107.82604800000001|ORD|43.0    |52.19846106422065|
|107.82604800000001|ORD|43.0    |52.19846106422065|
|107.82604800000001|ORD|43.0    |52.19846106422065|
|107.82604800000001|ORD|43.0    |52.19846106422065|
|107.82604800000001|ORD|43.0    |52.19846106422065|
|107.82604800000001|ORD|43.0    |52.19846106422065|
|107.82604800000001|ORD|44.0    |52.19846106422065|
|107.82604800000001|ORD|44.0    |52.19846106422065|
|107.82604800000001|ORD|46.0    |52.19846106422065|
|107.82604800000001|ORD|46.0    |52.19846106422065|
+------------------+---+--------+-----------------+
only showing top 10 rows
RMSE test: 11.2119 phút


**Pipeline chạy thành công**, đạt **RMSE test 11,2119 phút**, trùng với mô hình `km + org` ở Bài 6 do sử dụng cùng đặc trưng và cách chia dữ liệu.

Trong 10 dòng hiển thị, các chuyến đều có **cùng khoảng cách và sân bay ORD**, nên mô hình dự đoán cùng giá trị **52,20 phút**. Thời lượng thực tế khác nhau từ 43–46 phút cho thấy hai đặc trưng này chưa giải thích hết biến động giữa các chuyến bay.

RMSE được tính trên **toàn bộ tập test**, không chỉ 10 dòng trên. Các dòng giống nhau cũng chưa đủ để kết luận dữ liệu trùng, vì có thể là những chuyến bay khác ngày.

### Bài 9 - Cross Validation

Grid:

```text
regParam = [0.01, 0.1, 1.0]
elasticNetParam = [0, 0.5, 1]
```

1. có bao nhiêu model configuration?
2. chạy 3-fold CV;
3. tìm model tốt nhất;
4. đánh giá trên test set.

In [71]:
import time
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator

# 1. Grid: 9 cấu hình
param_grid = (
    ParamGridBuilder()
    .addGrid(lr.regParam, [0.01, 0.1, 1.0])
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0])
    .build()
)

print("Số cấu hình:", len(param_grid))

# RMSE càng thấp càng tốt
evaluator = RegressionEvaluator(
    labelCol="duration",
    predictionCol="prediction",
    metricName="rmse"
)

# 2. Cross Validation trên training data
cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=3,
    seed=42,
    parallelism=1
)

print("Đang chạy 3-fold CV...", flush=True)
start = time.perf_counter()

cv_model = cv.fit(reg_train)

print(f"Thời gian CV: {time.perf_counter() - start:.1f} giây")

# Kết quả trung bình của từng cấu hình qua 3 folds
print(f"\n{'regParam':>10} {'elasticNetParam':>17} {'CV RMSE':>12}")

for params, score in sorted(
    zip(param_grid, cv_model.avgMetrics),
    key=lambda item: item[1]
):
    print(
        f"{params[lr.regParam]:>10.2f} "
        f"{params[lr.elasticNetParam]:>17.2f} "
        f"{score:>12.4f}"
    )

# 3. Model tốt nhất đã được refit trên toàn bộ reg_train
best_pipeline_model = cv_model.bestModel
best_lr = best_pipeline_model.stages[-1]

print("\nCẤU HÌNH TỐT NHẤT")
print("regParam:", best_lr.getRegParam())
print("elasticNetParam:", best_lr.getElasticNetParam())
print(f"CV RMSE trung bình: {min(cv_model.avgMetrics):.4f}")

# 4. Đánh giá trên test, không dùng test để tuning
test_predictions = best_pipeline_model.transform(reg_test)
test_rmse = evaluator.evaluate(test_predictions)

print(f"\nRMSE test: {test_rmse:.4f} phút")

test_predictions.select(
    "km", "org", "duration", "prediction"
).show(10, truncate=False)

Số cấu hình: 9
Đang chạy 3-fold CV...
Thời gian CV: 25.9 giây

  regParam   elasticNetParam      CV RMSE
      0.01              0.50      11.2089
      0.01              0.00      11.2089
      0.01              1.00      11.2089
      0.10              0.00      11.2093
      0.10              0.50      11.2101
      0.10              1.00      11.2123
      1.00              0.00      11.2506
      1.00              0.50      11.3261
      1.00              1.00      11.5431

CẤU HÌNH TỐT NHẤT
regParam: 0.01
elasticNetParam: 0.5
CV RMSE trung bình: 11.2089

RMSE test: 11.2119 phút
+------------------+---+--------+-----------------+
|km                |org|duration|prediction       |
+------------------+---+--------+-----------------+
|107.82604800000001|ORD|43.0    |52.19515976004317|
|107.82604800000001|ORD|43.0    |52.19515976004317|
|107.82604800000001|ORD|43.0    |52.19515976004317|
|107.82604800000001|ORD|43.0    |52.19515976004317|
|107.82604800000001|ORD|43.0    |52.195159760

Cross Validation chọn Elastic Net với regParam = 0,01 và elasticNetParam = 0,5, đạt RMSE trung bình 11,2089 phút.
- RMSE test = 11,2119 phút, gần với CV, cho thấy kết quả khá nhất quán giữa các tập đánh giá.
- Ba cấu hình có regParam = 0,01 đều đạt 11,2089 khi làm tròn; vì vậy chưa thể khẳng định Elastic Net vượt trội rõ rệt so với Ridge hoặc Lasso.
- Khi tăng regParam lên 1,0, RMSE tăng, đặc biệt với Lasso (11,5431), cho thấy mức phạt mạnh làm giảm chất lượng dự báo.
- So với Linear Regression ban đầu (11,2119 phút trên test), tuning chưa tạo ra cải thiện đáng kể.
Có thể giữ cấu hình được CV chọn, nhưng mô hình Linear Regression đơn giản vẫn là lựa chọn hợp lý vì hiệu quả dự báo gần như tương đương.

### Bài 10 - Ensemble

So sánh:

- Decision Tree;
- Random Forest;
- GBT.

Chọn mô hình dựa trên AUC và F1.

In [72]:
from pyspark.ml.classification import (
    DecisionTreeClassifier,
    RandomForestClassifier,
    GBTClassifier
)

ensemble_classifiers = {
    "Decision Tree": DecisionTreeClassifier(
        maxDepth=5,
        seed=42
    ),
    "Random Forest": RandomForestClassifier(
        numTrees=30,
        maxDepth=5,
        seed=42
    ),
    "GBT": GBTClassifier(
        maxIter=20,
        maxDepth=4,
        seed=42
    )
}

ensemble_models = {}
ensemble_results = []

for name, classifier in ensemble_classifiers.items():
    print(f"Đang huấn luyện {name}...", flush=True)

    fitted = make_pipeline(classifier).fit(train)
    metrics = evaluate(fitted, valid)

    ensemble_models[name] = fitted
    ensemble_results.append({
        "model": name,
        "auc": metrics["auc"],
        "f1_delay": metrics["f1_delay"],
        "f1_weighted": metrics["f1_weighted"]
    })

# So sánh
ranked = sorted(
    ensemble_results,
    key=lambda r: (r["auc"], r["f1_delay"]),
    reverse=True
)

print(
    f"\n{'Model':<20} {'AUC':>10} "
    f"{'F1-delay':>12} {'F1-weighted':>14}"
)

for r in ranked:
    print(
        f"{r['model']:<20} "
        f"{r['auc']:>10.4f} "
        f"{r['f1_delay']:>12.4f} "
        f"{r['f1_weighted']:>14.4f}"
    )

best = ranked[0]
best_ensemble_model = ensemble_models[best["model"]]

print(
    f"\nChọn {best['model']}: "
    f"AUC={best['auc']:.4f}, "
    f"F1-delay={best['f1_delay']:.4f}."
)

Đang huấn luyện Decision Tree...
Đang huấn luyện Random Forest...
Đang huấn luyện GBT...

Model                       AUC     F1-delay    F1-weighted
GBT                      0.7063       0.6621         0.6545
Random Forest            0.6789       0.6465         0.6253
Decision Tree            0.5902       0.6423         0.6246

Chọn GBT: AUC=0.7063, F1-delay=0.6621.


**GBT là mô hình tốt nhất trong ba cấu hình đã thử**, đạt **AUC = 0,7063**, **F1-delay = 0,6621** và **F1-weighted = 0,6545**, đều cao nhất.

Random Forest cải thiện AUC so với Decision Tree, nhưng F1-delay chỉ tăng nhẹ. GBT cải thiện cả hai chỉ số, cho thấy khả năng phân biệt hai lớp và cân bằng Precision–Recall tốt hơn tại ngưỡng dự đoán hiện tại.

Vì vậy, **chọn GBT**, dù hiệu quả còn ở mức vừa phải và vẫn có thể cải thiện bằng tuning hoặc bổ sung đặc trưng.

<a id="summary"></a>
## Tổng kết

```text
Spark / DataFrame
       ↓
Data Preparation
       ↓
StringIndexer / OneHotEncoder
       ↓
VectorAssembler
       ↓
Classification
  ├─ Decision Tree
  └─ Logistic Regression
       ↓
Evaluation
       ↓
Text → TF-IDF
       ↓
Regression
       ↓
Feature Engineering
       ↓
Regularization
       ↓
Pipeline
       ↓
Cross Validation + Grid Search
       ↓
Ensemble Models
```

## Những điểm cần nhớ

1. Spark ML dùng `features` vector và `label`.
2. `StringIndexer` không phải One-Hot Encoding.
3. `VectorAssembler` ghép predictor thành một vector.
4. Preprocessing phải **fit trên training data**, không fit trên toàn bộ dataset.
5. Classification không nên chỉ nhìn Accuracy; cần Precision, Recall, F1, AUC.
6. Regression nên xem RMSE/MAE/R².
7. Lasso có thể tạo coefficient bằng 0.
8. Pipeline giúp tái sử dụng và giảm nguy cơ leakage.
9. Cross Validation dùng cho model selection/tuning; test set dành cho đánh giá cuối cùng.
10. Ensemble thường mạnh hơn một cây đơn nhưng tốn tài nguyên hơn.

In [73]:
# Khi hoàn thành toàn bộ bài:
spark.stop()